# Merge and Prepare the Dudley Datasets

## Purpose
Combine the five cleaned datasets to explore local factors associated
with economic activity and possible employment support needs in Dudley.

## Objectives
- Verify geographic compatibility between the datasets.
- Select relevant columns and distinguish their source years.
- Merge compatible areas using LSOA codes.
- Check merge coverage, missing values and duplicate records.
- Calculate percentages using the appropriate population or
  household denominator from each source.
- Save the combined dataset for later analysis.

## Inputs
Five cleaned CSV files from `cleaned_datasets`:
- `dudley_economic_activity_cleaned.csv`
- `dudley_qualifications_cleaned.csv`
- `dudley_disability_16_cleaned.csv`
- `dudley_car_availability_cleaned.csv`
- `dudley_iod_cleaned.csv`

## Planned Output
`cleaned_datasets/dudley_combined_prepared.csv`

## Geographic and Time Coverage
The Census datasets contain 203 Dudley areas. The deprivation
dataset contains 201 areas using 2011 geography.

The datasets share 199 LSOA codes. The planned merge will use
matching areas after boundary compatibility has been checked.
Any exclusions will be documented.

Deprivation data comes from the 2019 release and provides historical
context alongside Census 2021 data.

## Ethical Considerations
Area-level patterns do not establish individual circumstances or
prove causes of unemployment. Disability, qualifications and car
availability must be interpreted respectfully and in context.

Different source dates, population definitions and excluded areas
will be considered when interpreting results. The original cleaned
datasets will remain unchanged.

## Additional Comments
ChatGPT assisted with planning, code development and documentation.
This notebook covers merging and preparation; EDA, hypothesis
testing and modelling will follow separately.

---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'c:\\Dudley employment barriers\\jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [3]:
current_dir = os.getcwd()
current_dir

'c:\\Dudley employment barriers'

## 1. Load the Cleaned Datasets

Load each cleaned CSV into a separate dataframe.
Display their dimensions and column names to confirm the available
fields before selecting variables or calculating percentages.

No source files will be changed.

In [4]:
# Developed with ChatGPT assistance to load the cleaned datasets.
import pandas as pd

# Map each dataset to its saved CSV filename.
dataset_files = {
    "economic": (
        "dudley_economic_activity_cleaned.csv"
    ),
    "qualifications": (
        "dudley_qualifications_cleaned.csv"
    ),
    "disability": (
        "dudley_disability_16_cleaned.csv"
    ),
    "car": (
        "dudley_car_availability_cleaned.csv"
    ),
    "iod": "dudley_iod_cleaned.csv"
}

# Load each file separately, preserving the saved source files.
datasets = {
    name: pd.read_csv(
        f"cleaned_datasets/{filename}"
    )
    for name, filename in dataset_files.items()
}

# Inspect dimensions and exact column names before merging.
for name, dataframe in datasets.items():
    print(f"\n{name.upper()}")
    print(
        f"Rows: {dataframe.shape[0]}, "
        f"Columns: {dataframe.shape[1]}"
    )
    print(dataframe.columns.tolist())


ECONOMIC
Rows: 203, Columns: 11
['Original_Area', 'Total_Aged_16_Plus', 'Economically_Active_Excl_FT_Students', 'Employed_Excl_FT_Students', 'Unemployed_Excl_FT_Students', 'Inactive_Student', 'Inactive_Home_Or_Family', 'Inactive_Long_Term_Sick_Or_Disabled', 'Inactive_Other', 'LSOA_Code', 'LSOA_Name']

QUALIFICATIONS
Rows: 203, Columns: 10
['LSOA_Name', 'LSOA_Code', 'Total_Aged_16_Plus', 'No_Qualifications', 'Level_1_And_Entry', 'Level_2', 'Apprenticeship', 'Level_3', 'Level_4_And_Above', 'Other_Qualifications']

DISABILITY
Rows: 203, Columns: 7
['LSOA_Name', 'LSOA_Code', 'Total_16_Plus', 'Disabled_16_Plus', 'Limited_A_Lot_16_Plus', 'Limited_A_Little_16_Plus', 'Not_Disabled_16_Plus']

CAR
Rows: 203, Columns: 7
['LSOA_Name', 'LSOA_Code', 'Total_Households', 'No_Car_Van_Households', 'One_Car_Van_Households', 'Two_Cars_Vans_Households', 'Three_Plus_Cars_Vans_Households']

IOD
Rows: 201, Columns: 57
['LSOA_code_2011', 'LSOA_name_2011', 'Local_Authority_District_code_2019', 'Local_Authority

### Findings
- All five cleaned datasets loaded successfully.
- Economic activity contains 203 rows and 11 columns.
- Qualifications contains 203 rows and 10 columns.
- Disability and car availability each contain 203 rows and 7 columns.
- Deprivation contains 201 rows and 57 columns.
- The four Census datasets use `LSOA_Code`; deprivation uses
  `LSOA_code_2011`.
- Population totals have overlapping names across datasets.
  They must remain distinguishable so percentages use the
  correct source-specific denominator.
- The economic activity file contains selected categories,
  including employment measures excluding full-time students.
  These must not be labelled as covering all employment.
- No datasets have been merged or modified.

---

## 2. Check Merge Keys and Shared Coverage

Check that each dataset has complete, unique LSOA codes so merging
will not create duplicate area records.

Confirm whether the four Census datasets contain the same codes
and count the codes shared by all five datasets.

Matching codes establish join coverage. Compatibility between
2011 and 2021 boundaries still requires a separate check.

In [5]:
# Developed with ChatGPT assistance to check merge keys and coverage.

# Identify the geographic key used by each dataset.
key_columns = {
    name: (
        "LSOA_code_2011"
        if name == "iod"
        else "LSOA_Code"
    )
    for name in datasets
}

# Check missing and duplicate keys before merging.
# References:
# https://pandas.pydata.org/docs/reference/api/pandas.Series.isna.html
# https://pandas.pydata.org/docs/reference/api/pandas.Series.duplicated.html
check_results = []

for name, dataframe in datasets.items():
    codes = dataframe[key_columns[name]]

    check_results.append({
        "Dataset": name,
        "Rows": len(dataframe),
        "Missing_codes": codes.isna().sum(),
        "Duplicate_codes": codes.duplicated().sum()
    })

key_checks = pd.DataFrame(check_results)
display(key_checks)

# Remove missing codes and create sets of unique codes.
# References:
# https://pandas.pydata.org/docs/reference/api/pandas.Series.dropna.html
# https://docs.python.org/3/library/stdtypes.html#set-types-set-frozenset
code_sets = {}

for name, dataframe in datasets.items():
    codes = dataframe[key_columns[name]].dropna()
    code_sets[name] = set(codes)

# Check whether the Census datasets cover the same areas.
# Reference:
# https://docs.python.org/3/library/functions.html#all
census_names = [
    "economic",
    "qualifications",
    "disability",
    "car"
]

same_census_codes = all(
    code_sets[name] == code_sets["economic"]
    for name in census_names
)

# Find codes present in every dataset.
# Reference:
# https://docs.python.org/3/library/stdtypes.html#frozenset.intersection
shared_codes = set.intersection(
    *code_sets.values()
)

print(
    "All four Census code sets match:",
    same_census_codes
)
print(
    "Codes shared by all five datasets:",
    len(shared_codes)
)

,Dataset,Rows,Missing_codes,Duplicate_codes
0,economic,203,0,0
1,qualifications,203,0,0
2,disability,203,0,0
3,car,203,0,0
4,iod,201,0,0


All four Census code sets match: True
Codes shared by all five datasets: 199


### Findings
- All five datasets have complete, unique LSOA codes.
- No missing or duplicate codes were found.
- The four Census datasets contain the same 203 area codes.
- A total of 199 codes are shared by all five datasets.
- The keys support one-to-one joins without duplicate matches.
- Compatibility between the 2011 and 2021 boundaries still
  needs verification before merging.
- No records have been removed or merged.

---

## 3. Load the Geographic Lookup

Load the official ONS lookup linking 2011 and 2021 LSOA areas.
Inspect its columns and preview five records before using its
change indicator to check boundary compatibility.

This lookup supports geographic validation; it will not add
analytical features to the combined dataset.

In [6]:
# Developed with ChatGPT assistance to inspect the ONS lookup.

# Load the downloaded lookup without changing its contents.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html
lookup_path = "raw_datasets/lsoa_2011_2021_lookup.csv"
lookup_df = pd.read_csv(lookup_path)

print(f"Rows: {lookup_df.shape[0]}")
print(f"Columns: {lookup_df.shape[1]}")

# Display exact column names to guide the next check.
for column in lookup_df.columns:
    print(column)

# Preview five records.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.head.html
display(lookup_df.head())

Rows: 35796
Columns: 9
LSOA11CD
LSOA11NM
LSOA21CD
LSOA21NM
CHGIND
LAD22CD
LAD22NM
LAD22NMW
ObjectId


,LSOA11CD,LSOA11NM,LSOA21CD,LSOA21NM,CHGIND,LAD22CD,LAD22NM,LAD22NMW,ObjectId
0,E01031349,Adur 001A,E01031349,Adur 001A,U,E07000223,Adur,NaN,1
1,E01031350,Adur 001B,E01031350,Adur 001B,U,E07000223,Adur,NaN,2
2,E01031351,Adur 001C,E01031351,Adur 001C,U,E07000223,Adur,NaN,3
3,E01031352,Adur 001D,E01031352,Adur 001D,U,E07000223,Adur,NaN,4
4,E01031370,Adur 001E,E01031370,Adur 001E,U,E07000223,Adur,NaN,5


### Findings
- The lookup loaded successfully with 35,796 rows and 9 columns.
- `LSOA11CD` and `LSOA21CD` contain the geographic codes
  for 2011 and 2021.
- `CHGIND` records the type of geographic change.
- The five previewed records have indicator `U` (unchanged)
  and identical codes in both years.
- These preview records are for Adur. Dudley's matching areas
  still need to be checked.

---

## 4. Verify Boundary Compatibility for Shared Areas

Check the 199 shared codes against the ONS lookup.

Each shared code should have exactly one lookup record, the same
2011 and 2021 code, and change indicator `U` (unchanged).
Report any exceptions before merging.

In [7]:
# Developed with ChatGPT assistance to verify area boundaries.

# Select lookup records for the shared 2011 codes.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.Series.isin.html
shared_lookup = lookup_df.loc[
    lookup_df["LSOA11CD"].isin(shared_codes)
].copy()

# Check that every shared code has exactly one lookup record.
# References:
# https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html
# https://pandas.pydata.org/docs/reference/api/pandas.Series.reindex.html
lookup_counts = (
    shared_lookup["LSOA11CD"].value_counts()
)
lookup_counts = lookup_counts.reindex(
    sorted(shared_codes),
    fill_value=0
)

print("Shared codes checked:", len(shared_codes))
print(
    "Codes missing from lookup:",
    (lookup_counts == 0).sum()
)
print(
    "Codes with multiple records:",
    (lookup_counts > 1).sum()
)

# Flag changed areas or different codes between the two years.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.Series.eq.html
same_code = shared_lookup["LSOA11CD"].eq(
    shared_lookup["LSOA21CD"]
)
unchanged = shared_lookup["CHGIND"].eq("U")
exceptions = shared_lookup.loc[
    ~(same_code & unchanged)
]

print(
    "Records failing boundary checks:",
    len(exceptions)
)
print("\nChange indicators:")
print(
    shared_lookup["CHGIND"].value_counts(
        dropna=False
    )
)

# Preview any exceptions, showing only the relevant fields.
display(
    exceptions[
        ["LSOA11CD", "LSOA21CD", "CHGIND", "LAD22NM"]
    ].head()
)

Shared codes checked: 199
Codes missing from lookup: 0
Codes with multiple records: 0
Records failing boundary checks: 0

Change indicators:
CHGIND
U    199
Name: count, dtype: int64


,LSOA11CD,LSOA21CD,CHGIND,LAD22NM


### Findings
- All 199 shared codes were found in the official ONS lookup.
- Each code has exactly one lookup record.
- All records have identical 2011 and 2021 LSOA codes.
- All 199 records have change indicator `U` (unchanged).
- No boundary-check exceptions were found.
- The shared areas are geographically compatible for merging.
  Differences in dataset dates still apply.

---

## 5. Check Percentage Denominators

Check that each Census dataset's population or household total
is present and greater than zero before calculating percentages.

Use the total from the same dataset as the category being measured.
Do not substitute totals from another source, even when the
population labels are similar.

The deprivation rates are already supplied as proportions and
will be handled separately.

In [8]:
# Developed with ChatGPT assistance to check denominators.

# Use each dataset's own population or household total.
denominator_columns = {
    "economic": "Total_Aged_16_Plus",
    "qualifications": "Total_Aged_16_Plus",
    "disability": "Total_16_Plus",
    "car": "Total_Households"
}

# Check missing, zero and negative denominator values.
# References:
# https://pandas.pydata.org/docs/reference/api/pandas.Series.isna.html
# https://pandas.pydata.org/docs/reference/api/pandas.Series.le.html
denominator_results = []

for name, column in denominator_columns.items():
    totals = datasets[name][column]

    denominator_results.append({
        "Dataset": name,
        "Denominator": column,
        "Minimum": totals.min(),
        "Maximum": totals.max(),
        "Missing": totals.isna().sum(),
        "Zero_or_negative": totals.le(0).sum()
    })

denominator_checks = pd.DataFrame(denominator_results)
display(denominator_checks)

,Dataset,Denominator,Minimum,Maximum,Missing,Zero_or_negative
0,economic,Total_Aged_16_Plus,986,2166,0,0
1,qualifications,Total_Aged_16_Plus,985,2169,0,0
2,disability,Total_16_Plus,985,2168,0,0
3,car,Total_Households,460,1183,0,0


### Findings
- All four denominator columns contain positive values.
- No missing, zero or negative denominators were found.
- Population totals vary slightly between the Census datasets.
  Each percentage will therefore use its own source's total.
- Car availability percentages will use household totals,
  rather than population totals.
- These totals support calculating shares of residents aged
  16 and over, or households, as appropriate.
- An unemployment rate would require a separate economically
  active denominator; it is not the same as a population share.

---

## 6. Calculate Qualification Percentages

Calculate each qualification category as a percentage of residents
aged 16 and over, using the qualifications dataset's own total.

Retain the original counts and add percentage columns in a separate
working copy. Keep full precision for later analysis.

In [9]:
# Developed with ChatGPT assistance to calculate qualification shares.

# Create a working copy, keeping the loaded source unchanged.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.copy.html
qualifications_prepared = (
    datasets["qualifications"].copy()
)

qualification_columns = [
    "No_Qualifications",
    "Level_1_And_Entry",
    "Level_2",
    "Apprenticeship",
    "Level_3",
    "Level_4_And_Above",
    "Other_Qualifications"
]

# Divide each category by its source population total.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.Series.div.html
population = qualifications_prepared[
    "Total_Aged_16_Plus"
]

for column in qualification_columns:
    percentage_column = f"{column}_Pct"
    qualifications_prepared[percentage_column] = (
        qualifications_prepared[column]
        .div(population)
        * 100
    )

# Check that all calculated percentages fall between 0 and 100.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.Series.between.html
qualification_pct_columns = [
    f"{column}_Pct"
    for column in qualification_columns
]

invalid_percentages = qualifications_prepared[
    qualification_pct_columns
].apply(
    lambda column: (~column.between(0, 100)).sum()
)

print(
    "Percentage columns added:",
    len(qualification_pct_columns)
)
print(
    "Values outside 0–100:",
    invalid_percentages.sum()
)

# Preview five areas; round only the displayed values.
preview_columns = (
    ["LSOA_Name"] + qualification_pct_columns
)

display(
    qualifications_prepared[preview_columns]
    .head()
    .round(2)
)

Percentage columns added: 7
Values outside 0–100: 0


,LSOA_Name,No_Qualifications_Pct,Level_1_And_Entry_Pct,Level_2_Pct,Apprenticeship_Pct,Level_3_Pct,Level_4_And_Above_Pct,Other_Qualifications_Pct
0,Dudley 001A,26.63,11.18,17.65,6.23,17.47,18.08,2.75
1,Dudley 001B,24.17,10.43,14.77,7.74,16.90,22.04,3.95
2,Dudley 001C,28.30,11.54,17.39,5.77,14.31,18.81,3.87
3,Dudley 001D,32.70,10.99,17.09,6.16,14.78,15.48,2.80
4,Dudley 001E,32.10,12.51,15.47,5.93,14.90,16.30,2.80


### Findings
- Seven qualification percentage columns were added.
- Each percentage uses the qualifications dataset's own
  population aged 16 and over as its denominator.
- No calculated percentages fall outside 0–100.
- In Dudley 001A, 26.63% of residents aged 16 and over have
  no qualifications, while 18.08% have Level 4 or above.
- Original counts are retained in `qualifications_prepared`.
- Full precision is preserved; only the preview is rounded
  to two decimal places.
- The original cleaned CSV has not been changed.

---

## 7. Calculate Disability Percentages

Calculate disability categories as percentages of all residents
aged 16 and over, using the disability dataset's own total.

The limited-a-lot and limited-a-little percentages also use all
residents aged 16 and over, not only disabled residents.
These two categories are subsets of the disabled category.

Retain the original counts and preserve full calculation precision.

In [10]:
# Developed with ChatGPT assistance to calculate disability shares.

# Create a separate working copy.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.copy.html
disability_prepared = datasets["disability"].copy()

disability_columns = [
    "Disabled_16_Plus",
    "Limited_A_Lot_16_Plus",
    "Limited_A_Little_16_Plus",
    "Not_Disabled_16_Plus"
]

# Calculate each category's share of all residents aged 16+.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.Series.div.html
disability_population = disability_prepared[
    "Total_16_Plus"
]

for column in disability_columns:
    disability_prepared[f"{column}_Pct"] = (
        disability_prepared[column]
        .div(disability_population)
        * 100
    )

# Check that percentages fall between 0 and 100.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.Series.between.html
disability_pct_columns = [
    f"{column}_Pct"
    for column in disability_columns
]

invalid_disability_pct = disability_prepared[
    disability_pct_columns
].apply(
    lambda column: (~column.between(0, 100)).sum()
)

print(
    "Percentage columns added:",
    len(disability_pct_columns)
)
print(
    "Values outside 0–100:",
    invalid_disability_pct.sum()
)

# Preview five areas, rounding only the displayed values.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.round.html
preview_columns = (
    ["LSOA_Name"] + disability_pct_columns
)

display(
    disability_prepared[preview_columns]
    .head()
    .round(2)
)

Percentage columns added: 4
Values outside 0–100: 0


,LSOA_Name,Disabled_16_Plus_Pct,Limited_A_Lot_16_Plus_Pct,Limited_A_Little_16_Plus_Pct,Not_Disabled_16_Plus_Pct
0,Dudley 001A,21.34,9.39,11.95,78.66
1,Dudley 001B,20.65,9.57,11.08,79.35
2,Dudley 001C,23.34,11.44,11.91,76.66
3,Dudley 001D,26.33,14.57,11.76,73.67
4,Dudley 001E,22.92,11.21,11.71,77.08


### Findings
- Four disability percentage columns were added.
- All percentages use the disability dataset's total population
  aged 16 and over.
- No calculated percentages fall outside 0–100.
- In Dudley 001A, 21.34% of residents aged 16 and over are
  disabled: 9.39% have activities limited a lot and 11.95%
  have activities limited a little.
- The limitation categories are subsets of the disabled category
  and must not be added to it as separate groups.
- Original counts and full calculation precision are retained.
  Only the preview is rounded to two decimal places.
- The original cleaned CSV has not been changed.

---

## 8. Calculate Car Availability Percentages

Calculate each car or van availability category as a percentage
of households, using the car availability dataset's own total.

These percentages describe households, not individual residents.
Having no car or van does not by itself establish an inability
to travel to work.

Retain the original counts and full calculation precision.

In [11]:
# Developed with ChatGPT assistance to calculate household shares.

# Create a separate working copy.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.copy.html
car_prepared = datasets["car"].copy()

car_columns = [
    "No_Car_Van_Households",
    "One_Car_Van_Households",
    "Two_Cars_Vans_Households",
    "Three_Plus_Cars_Vans_Households"
]

# Calculate each category's share of all households.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.Series.div.html
household_total = car_prepared["Total_Households"]

for column in car_columns:
    car_prepared[f"{column}_Pct"] = (
        car_prepared[column]
        .div(household_total)
        * 100
    )

# Check that percentages fall between 0 and 100.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.Series.between.html
car_pct_columns = [
    f"{column}_Pct"
    for column in car_columns
]

invalid_car_pct = car_prepared[
    car_pct_columns
].apply(
    lambda column: (~column.between(0, 100)).sum()
)

print(
    "Percentage columns added:",
    len(car_pct_columns)
)
print(
    "Values outside 0–100:",
    invalid_car_pct.sum()
)

# Preview five areas, rounding only the displayed values.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.round.html
preview_columns = ["LSOA_Name"] + car_pct_columns

display(
    car_prepared[preview_columns]
    .head()
    .round(2)
)

Percentage columns added: 4
Values outside 0–100: 0


,LSOA_Name,No_Car_Van_Households_Pct,One_Car_Van_Households_Pct,Two_Cars_Vans_Households_Pct,Three_Plus_Cars_Vans_Households_Pct
0,Dudley 001A,29.59,39.00,23.47,7.94
1,Dudley 001B,12.03,41.77,34.02,12.18
2,Dudley 001C,23.10,41.64,26.29,8.97
3,Dudley 001D,40.78,38.59,14.68,5.95
4,Dudley 001E,23.86,43.01,24.65,8.48


### Findings
- Four car or van availability percentage columns were added.
- Each percentage uses the dataset's total households as
  its denominator.
- No calculated percentages fall outside 0–100.
- In Dudley 001A, 29.59% of households have no car or van,
  while 39.00% have one.
- These are household shares, not percentages of residents.
- Original counts and full calculation precision are retained.
  Only the preview is rounded to two decimal places.
- The original cleaned CSV has not been changed.

---

## 9. Calculate Economic Activity Population Shares

Calculate the selected economic activity categories as percentages
of all residents aged 16 and over, using this dataset's own total.

Employment, unemployment and economic activity counts exclude
full-time students, as indicated in their column names.

The unemployment percentage calculated here is a share of all
residents aged 16 and over, not an unemployment rate.

The available inactivity categories do not cover every reason
for inactivity, so they will not be summed into a total.
Some columns overlap and must not be added together.

In [12]:
# Developed with ChatGPT assistance to calculate population shares.

# Create a separate working copy.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.copy.html
economic_prepared = datasets["economic"].copy()

economic_columns = [
    "Economically_Active_Excl_FT_Students",
    "Employed_Excl_FT_Students",
    "Unemployed_Excl_FT_Students",
    "Inactive_Student",
    "Inactive_Home_Or_Family",
    "Inactive_Long_Term_Sick_Or_Disabled",
    "Inactive_Other"
]

# Express each selected category as a share of all residents 16+.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.Series.div.html
economic_population = economic_prepared[
    "Total_Aged_16_Plus"
]

for column in economic_columns:
    economic_prepared[f"{column}_Pct"] = (
        economic_prepared[column]
        .div(economic_population)
        * 100
    )

# Check that percentages fall between 0 and 100.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.Series.between.html
economic_pct_columns = [
    f"{column}_Pct"
    for column in economic_columns
]

invalid_economic_pct = economic_prepared[
    economic_pct_columns
].apply(
    lambda column: (~column.between(0, 100)).sum()
)

print(
    "Percentage columns added:",
    len(economic_pct_columns)
)
print(
    "Values outside 0–100:",
    invalid_economic_pct.sum()
)

# Preview five areas, rounding only the displayed values.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.round.html
preview_columns = (
    ["LSOA_Name"] + economic_pct_columns
)

display(
    economic_prepared[preview_columns]
    .head()
    .round(2)
)

Percentage columns added: 7
Values outside 0–100: 0


,LSOA_Name,Economically_Active_Excl_FT_Students_Pct,Employed_Excl_FT_Students_Pct,Unemployed_Excl_FT_Students_Pct,Inactive_Student_Pct,Inactive_Home_Or_Family_Pct,Inactive_Long_Term_Sick_Or_Disabled_Pct,Inactive_Other_Pct
0,Dudley 001A,59.25,54.25,5.01,5.93,10.02,4.46,4.28
1,Dudley 001B,60.05,57.83,2.22,2.77,3.96,2.77,2.45
2,Dudley 001C,59.13,55.26,3.87,2.77,5.45,5.22,2.21
3,Dudley 001D,48.99,43.19,5.80,5.45,8.32,10.34,4.12
4,Dudley 001E,54.70,51.48,3.21,3.38,6.01,5.68,4.12


### Findings
- Seven economic activity percentage columns were added.
- Each percentage uses the economic activity dataset's total
  population aged 16 and over.
- No calculated percentages fall outside 0–100.
- In Dudley 001A, employed residents excluding full-time students
  represent 54.25% of the population aged 16 and over.
- Unemployed residents excluding full-time students represent
  5.01% of that population. This is a population share,
  not an unemployment rate.
- No overall inactivity percentage was calculated because the
  available inactivity categories are incomplete.
- Original counts and full calculation precision are retained.
  Only the preview is rounded to two decimal places.
- The original cleaned CSV has not been changed.

---

## 10. Prepare Columns for Merging

Keep one LSOA code column for joining and retain the area name
from the economic activity dataset.

Give population totals distinct names so their sources remain
clear. Prefix deprivation measures with `IoD2019_` to identify
their release year.

Remove repeated area-name fields and the original economic area
label from the merge copies. Preserve the prepared source tables.

In [13]:
# Developed with ChatGPT assistance to prepare merge columns.

# Remove redundant labels and distinguish population totals.
# References:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop.html
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rename.html
economic_merge = economic_prepared.drop(
    columns=["Original_Area"]
).rename(
    columns={
        "Total_Aged_16_Plus": "Economic_Total_16_Plus"
    }
)

qualifications_merge = qualifications_prepared.drop(
    columns=["LSOA_Name"]
).rename(
    columns={
        "Total_Aged_16_Plus": (
            "Qualifications_Total_16_Plus"
        )
    }
)

disability_merge = disability_prepared.drop(
    columns=["LSOA_Name"]
).rename(
    columns={
        "Total_16_Plus": "Disability_Total_16_Plus"
    }
)

car_merge = car_prepared.drop(
    columns=["LSOA_Name"]
)

# Standardise the deprivation join key and label its release year.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rename.html
iod_merge = datasets["iod"].drop(
    columns=["LSOA_name_2011"]
)

iod_names = {
    column: (
        "LSOA_Code"
        if column == "LSOA_code_2011"
        else f"IoD2019_{column}"
    )
    for column in iod_merge.columns
}
iod_merge = iod_merge.rename(columns=iod_names)

# Collect the merge copies and check for overlapping field names.
merge_tables = {
    "economic": economic_merge,
    "qualifications": qualifications_merge,
    "disability": disability_merge,
    "car": car_merge,
    "iod": iod_merge
}

# Only LSOA_Code should appear in more than one merge table.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html
all_columns = [
    column
    for dataframe in merge_tables.values()
    for column in dataframe.columns
    if column != "LSOA_Code"
]

column_counts = pd.Series(all_columns).value_counts()
overlapping_columns = column_counts[
    column_counts > 1
]

print(
    "Overlapping non-key column names:",
    len(overlapping_columns)
)
print(overlapping_columns.to_string())

for name, dataframe in merge_tables.items():
    print(
        f"{name}: {dataframe.shape[0]} rows, "
        f"{dataframe.shape[1]} columns"
    )

Overlapping non-key column names: 0
Series([], )
economic: 203 rows, 17 columns
qualifications: 203 rows, 16 columns
disability: 203 rows, 10 columns
car: 203 rows, 10 columns
iod: 201 rows, 56 columns


### Findings
- All five merge tables use `LSOA_Code` as their join key.
- No overlapping non-key column names were found.
- Population totals have distinct source-specific names.
- Deprivation fields carry the `IoD2019_` prefix.
- The economic activity table retains the area name;
  redundant area labels were removed from the merge copies.
- All source rows remain available: 203 in each Census table
  and 201 in the deprivation table.
- No merging or geographic exclusions have taken place yet.

---

## 11. Merge the Five Datasets

Join the prepared tables using `LSOA_Code`, retaining only areas
present in every dataset.

Validate each join as one-to-one to prevent duplicate area matches.
The expected result is 199 areas whose unchanged boundaries have
already been confirmed through the ONS lookup.

Record row counts after each join and check the final code coverage.
The separate source tables remain unchanged.

In [14]:
# Developed with ChatGPT assistance to merge the prepared datasets.

# Start with a copy of the economic activity table.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.copy.html
combined_df = economic_merge.copy()
merge_results = []

# Retain matching codes and enforce one-to-one joins.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html
merge_order = [
    "qualifications",
    "disability",
    "car",
    "iod"
]

for name in merge_order:
    rows_before = len(combined_df)

    combined_df = combined_df.merge(
        merge_tables[name],
        on="LSOA_Code",
        how="inner",
        validate="one_to_one"
    )

    merge_results.append({
        "Dataset_added": name,
        "Rows_before": rows_before,
        "Rows_after": len(combined_df),
        "Rows_excluded": (
            rows_before - len(combined_df)
        )
    })

display(pd.DataFrame(merge_results))

# Confirm the merged codes match the verified shared set.
# Reference:
# https://docs.python.org/3/library/stdtypes.html#set-types-set-frozenset
merged_codes = set(combined_df["LSOA_Code"])

print("Final rows:", combined_df.shape[0])
print("Final columns:", combined_df.shape[1])
print(
    "Codes match verified shared areas:",
    merged_codes == shared_codes
)

# Preview five area identifiers.
display(
    combined_df[["LSOA_Code", "LSOA_Name"]]
    .head()
)

,Dataset_added,Rows_before,Rows_after,Rows_excluded
0,qualifications,203,203,0
1,disability,203,203,0
2,car,203,203,0
3,iod,203,199,4


Final rows: 199
Final columns: 105
Codes match verified shared areas: True


,LSOA_Code,LSOA_Name
0,E01009763,Dudley 001A
1,E01009764,Dudley 001B
2,E01009767,Dudley 001C
3,E01009769,Dudley 001D
4,E01009770,Dudley 001E


### Findings
- All four joins passed one-to-one validation.
- Joining the Census datasets retained all 203 areas.
- Adding deprivation data excluded four unmatched Census areas.
- The combined dataset contains 199 rows and 105 columns.
- Its LSOA codes exactly match the 199 shared areas whose
  unchanged boundaries were verified using the ONS lookup.
- The two deprivation-only areas were also excluded by the
  inner join.
- The combined dataset covers 199 of Dudley's 203 Census areas.
  Findings must acknowledge this reduced coverage.
- The separate source tables and cleaned CSVs remain unchanged.

---

## 12. Validate the Combined Dataset

Check the merged dataset for missing values, duplicate rows,
duplicate LSOA codes and duplicate column names.

Check that numeric values are finite and that the calculated
percentage columns remain within 0–100 after merging.

In [15]:
# Developed with ChatGPT assistance to validate the merged dataset.
import numpy as np

# Check completeness and duplicate records or identifiers.
# References:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.isna.html
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html
print(
    "Missing values:",
    combined_df.isna().sum().sum()
)
print(
    "Duplicate rows:",
    combined_df.duplicated().sum()
)
print(
    "Duplicate LSOA codes:",
    combined_df["LSOA_Code"].duplicated().sum()
)
print(
    "Duplicate column names:",
    combined_df.columns.duplicated().sum()
)

# Check numeric columns for infinite values.
# References:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.select_dtypes.html
# https://numpy.org/doc/stable/reference/generated/numpy.isinf.html
numeric_data = combined_df.select_dtypes(
    include="number"
)

print(
    "Infinite numeric values:",
    np.isinf(numeric_data).sum().sum()
)

# Check all calculated percentage columns after merging.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.Series.between.html
percentage_columns = [
    column
    for column in combined_df.columns
    if column.endswith("_Pct")
]

invalid_pct = combined_df[percentage_columns].apply(
    lambda column: (~column.between(0, 100)).sum()
)

print(
    "Percentage columns checked:",
    len(percentage_columns)
)
print(
    "Percentage values outside 0–100:",
    invalid_pct.sum()
)

Missing values: 0
Duplicate rows: 0
Duplicate LSOA codes: 0
Duplicate column names: 0
Infinite numeric values: 0
Percentage columns checked: 22
Percentage values outside 0–100: 0


### Findings
- No missing values or duplicate rows were found.
- LSOA codes and column names are unique.
- No infinite numeric values were found.
- All 22 calculated percentage columns were checked.
- No percentage values fall outside 0–100.
- The combined dataset passed these validation checks
  without requiring corrections.

---

## 13. Save the Combined Prepared Dataset

Save the validated dataset containing 199 areas and 105 columns
for use in later analysis.

Retain original counts, source-specific totals, calculated
percentages and historical deprivation measures. Export without
the dataframe index and preserve full numerical precision.

The five separate cleaned CSV files remain unchanged.

In [16]:
# Developed with ChatGPT assistance to save the prepared dataset.

# Export without an index or additional rounding.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html
output_path = "cleaned_datasets/dudley_combined_prepared.csv"
combined_df.to_csv(output_path, index=False)

# Report the saved dataset dimensions.
print(f"Saved: {output_path}")
print(f"Rows: {combined_df.shape[0]}")
print(f"Columns: {combined_df.shape[1]}")

Saved: cleaned_datasets/dudley_combined_prepared.csv
Rows: 199
Columns: 105


### Findings
- The prepared dataset was saved successfully to
  `cleaned_datasets/dudley_combined_prepared.csv`.
- It contains 199 rows and 105 columns.
- No dataframe index was exported or additional rounding applied.
- The five separate cleaned CSV files remain unchanged.

---

## 14. Create the Data Dictionary

Document each variable’s meaning, source, units and denominator so the combined dataset can be interpreted correctly. First inspect the saved dataset’s exact column names and data types, then use this inventory to build the dictionary.

In [17]:
# Read the saved dataset so the dictionary matches the exported file.
dictionary_data = pd.read_csv(
    "cleaned_datasets/dudley_combined_prepared.csv"
)

# Print every column and its type without pandas truncating the list.
for number, (column, dtype) in enumerate(
    dictionary_data.dtypes.items(), start=1
):
    print(f"{number}. {column} — {dtype}")

print(f"\nTotal columns: {len(dictionary_data.columns)}")

1. Economic_Total_16_Plus — int64
2. Economically_Active_Excl_FT_Students — int64
3. Employed_Excl_FT_Students — int64
4. Unemployed_Excl_FT_Students — int64
5. Inactive_Student — int64
6. Inactive_Home_Or_Family — int64
7. Inactive_Long_Term_Sick_Or_Disabled — int64
8. Inactive_Other — int64
9. LSOA_Code — object
10. LSOA_Name — object
11. Economically_Active_Excl_FT_Students_Pct — float64
12. Employed_Excl_FT_Students_Pct — float64
13. Unemployed_Excl_FT_Students_Pct — float64
14. Inactive_Student_Pct — float64
15. Inactive_Home_Or_Family_Pct — float64
16. Inactive_Long_Term_Sick_Or_Disabled_Pct — float64
17. Inactive_Other_Pct — float64
18. Qualifications_Total_16_Plus — int64
19. No_Qualifications — int64
20. Level_1_And_Entry — int64
21. Level_2 — int64
22. Apprenticeship — int64
23. Level_3 — int64
24. Level_4_And_Above — int64
25. Other_Qualifications — int64
26. No_Qualifications_Pct — float64
27. Level_1_And_Entry_Pct — float64
28. Level_2_Pct — float64
29. Apprenticeship_Pct 

### Column inventory findings

The saved combined dataset contains **105 columns**. These include geographic identifiers, Census counts and calculated percentages, historical IoD2019 scores, ranks and deciles, and population measures supplied with the deprivation data.

Population and household totals are source-specific and must not be treated as interchangeable denominators. Deprivation scores also use different scales; they must not all be described as percentages.

The `Selected_Inactive_Count` and `Selected_Inactive_Pct` variables created in notebook 08 are not stored in this prepared CSV. They will be documented separately as analysis-derived variables.

This inventory establishes the dictionary’s coverage. Variable definitions, sources, units and denominators still need to be added before the data-dictionary acceptance criterion is complete.

### 14.1 Census variables and geographic identifiers

Create one dictionary entry for each Census count, percentage and geographic identifier. Each percentage uses the total from its own source. Counts are not ratios, so their denominator field is marked not applicable; their population coverage is recorded separately.

The disability data come from the RM019 extract for residents aged 16+, while car availability concerns households. The dictionary records these distinctions to prevent inappropriate comparisons.

In [18]:
# Developed with ChatGPT assistance using verified column names.
# Document each count and percentage consistently.
dictionary_rows = []


def add_entry(
    column,
    meaning,
    source,
    units,
    population,
    denominator,
    calculation
):
    dictionary_rows.append({
        "Variable": column,
        "Data_Type": str(dictionary_data[column].dtype),
        "Meaning": meaning,
        "Source": source,
        "Reference_Year": "2021",
        "Units": units,
        "Population_Coverage": population,
        "Denominator": denominator,
        "Calculation": calculation
    })


# Identifiers are labels, not measurements.
for column, meaning in {
    "LSOA_Code": (
        "Lower-layer Super Output Area code "
        "used as the merge key."
    ),
    "LSOA_Name": (
        "Census 2021 Lower-layer Super Output Area name."
    )
}.items():
    add_entry(
        column,
        meaning,
        "ONS Census 2021 geography via Nomis",
        "Text identifier",
        "Included LSOAs",
        "Not applicable",
        "Retained from cleaned Census data"
    )

# Define source groups with their own denominators.
# Source references:
# https://www.nomisweb.co.uk/datasets/c2021ts066
# https://www.nomisweb.co.uk/datasets/c2021ts067
# https://www.nomisweb.co.uk/datasets/c2021rm019
# https://www.nomisweb.co.uk/datasets/c2021ts045
groups = [
    {
        "source": "ONS Census 2021 / Nomis TS066",
        "total": "Economic_Total_16_Plus",
        "population": (
            "All usual residents aged 16+ "
            "in the economic activity source"
        ),
        "units": "Persons",
        "variables": {
            "Economically_Active_Excl_FT_Students": (
                "Economically active residents, "
                "excluding full-time students"
            ),
            "Employed_Excl_FT_Students": (
                "Employed residents, "
                "excluding full-time students"
            ),
            "Unemployed_Excl_FT_Students": (
                "Unemployed residents, "
                "excluding full-time students"
            ),
            "Inactive_Student": (
                "Economically inactive students"
            ),
            "Inactive_Home_Or_Family": (
                "Economically inactive residents "
                "looking after home or family"
            ),
            "Inactive_Long_Term_Sick_Or_Disabled": (
                "Economically inactive residents "
                "who are long-term sick or disabled"
            ),
            "Inactive_Other": (
                "Economically inactive residents "
                "in the other category"
            )
        }
    },
    {
        "source": "ONS Census 2021 / Nomis TS067",
        "total": "Qualifications_Total_16_Plus",
        "population": (
            "All usual residents aged 16+ "
            "in the qualifications source"
        ),
        "units": "Persons",
        "variables": {
            "No_Qualifications": (
                "Residents with no formal qualifications"
            ),
            "Level_1_And_Entry": (
                "Residents whose highest qualification "
                "is level 1 or entry level"
            ),
            "Level_2": (
                "Residents whose highest qualification "
                "is level 2"
            ),
            "Apprenticeship": (
                "Residents classified in the "
                "apprenticeship qualification category"
            ),
            "Level_3": (
                "Residents whose highest qualification "
                "is level 3"
            ),
            "Level_4_And_Above": (
                "Residents whose highest qualification "
                "is level 4 or above"
            ),
            "Other_Qualifications": (
                "Residents classified in the "
                "other qualifications category"
            )
        }
    },
    {
        "source": (
            "ONS Census 2021 / Nomis RM019; "
            "economic activity filter: Total"
        ),
        "total": "Disability_Total_16_Plus",
        "population": (
            "All usual residents aged 16+ "
            "in the disability source"
        ),
        "units": "Persons",
        "variables": {
            "Disabled_16_Plus": (
                "Residents disabled under the Equality Act"
            ),
            "Limited_A_Lot_16_Plus": (
                "Disabled residents whose day-to-day "
                "activities are limited a lot"
            ),
            "Limited_A_Little_16_Plus": (
                "Disabled residents whose day-to-day "
                "activities are limited a little"
            ),
            "Not_Disabled_16_Plus": (
                "Residents not disabled "
                "under the Equality Act"
            )
        }
    },
    {
        "source": "ONS Census 2021 / Nomis TS045",
        "total": "Total_Households",
        "population": (
            "All households in the "
            "car or van availability source"
        ),
        "units": "Households",
        "variables": {
            "No_Car_Van_Households": (
                "Households with no car or van available"
            ),
            "One_Car_Van_Households": (
                "Households with one car or van available"
            ),
            "Two_Cars_Vans_Households": (
                "Households with two cars or vans available"
            ),
            "Three_Plus_Cars_Vans_Households": (
                "Households with three or more "
                "cars or vans available"
            )
        }
    }
]

# Document each total, count and derived percentage.
for group in groups:
    total = group["total"]

    add_entry(
        total,
        (
            "Source population total: "
            f"{group['population']}."
        ),
        group["source"],
        group["units"],
        group["population"],
        "Not applicable — count",
        "Retained source count"
    )

    for column, meaning in group["variables"].items():
        add_entry(
            column,
            meaning,
            group["source"],
            group["units"],
            group["population"],
            "Not applicable — count",
            "Retained source count"
        )

        add_entry(
            f"{column}_Pct",
            (
                f"{meaning}, expressed as a "
                "percentage of the source total."
            ),
            group["source"],
            "Percent (0–100)",
            group["population"],
            total,
            f"{column} / {total} * 100"
        )

# Check coverage against the actual CSV.
census_dictionary = pd.DataFrame(dictionary_rows)

expected_columns = [
    column
    for column in dictionary_data.columns
    if not column.startswith("IoD2019_")
]

missing_entries = sorted(
    set(expected_columns)
    - set(census_dictionary["Variable"])
)

unexpected_entries = sorted(
    set(census_dictionary["Variable"])
    - set(expected_columns)
)

duplicate_entries = (
    census_dictionary["Variable"].duplicated().sum()
)

print(
    "Census and geographic entries:",
    len(census_dictionary)
)
print("Missing entries:", missing_entries)
print("Unexpected entries:", unexpected_entries)
print("Duplicate entries:", duplicate_entries)

if (
    missing_entries
    or unexpected_entries
    or duplicate_entries
):
    raise ValueError(
        "Review dictionary coverage before continuing."
    )

# Match the CSV's column order.
census_dictionary = (
    census_dictionary.set_index("Variable")
    .loc[expected_columns]
    .reset_index()
)

display(census_dictionary.head(10))

Census and geographic entries: 50
Missing entries: []
Unexpected entries: []
Duplicate entries: 0


,Variable,Data_Type,Meaning,Source,Reference_Year,Units,Population_Coverage,Denominator,Calculation
0,Economic_Total_16_Plus,int64,Source population total: All usual residents a...,ONS Census 2021 / Nomis TS066,2021,Persons,All usual residents aged 16+ in the economic a...,Not applicable — count,Retained source count
1,Economically_Active_Excl_FT_Students,int64,"Economically active residents, excluding full-...",ONS Census 2021 / Nomis TS066,2021,Persons,All usual residents aged 16+ in the economic a...,Not applicable — count,Retained source count
2,Employed_Excl_FT_Students,int64,"Employed residents, excluding full-time students",ONS Census 2021 / Nomis TS066,2021,Persons,All usual residents aged 16+ in the economic a...,Not applicable — count,Retained source count
3,Unemployed_Excl_FT_Students,int64,"Unemployed residents, excluding full-time stud...",ONS Census 2021 / Nomis TS066,2021,Persons,All usual residents aged 16+ in the economic a...,Not applicable — count,Retained source count
4,Inactive_Student,int64,Economically inactive students,ONS Census 2021 / Nomis TS066,2021,Persons,All usual residents aged 16+ in the economic a...,Not applicable — count,Retained source count
5,Inactive_Home_Or_Family,int64,Economically inactive residents looking after ...,ONS Census 2021 / Nomis TS066,2021,Persons,All usual residents aged 16+ in the economic a...,Not applicable — count,Retained source count
6,Inactive_Long_Term_Sick_Or_Disabled,int64,Economically inactive residents who are long-t...,ONS Census 2021 / Nomis TS066,2021,Persons,All usual residents aged 16+ in the economic a...,Not applicable — count,Retained source count
7,Inactive_Other,int64,Economically inactive residents in the other c...,ONS Census 2021 / Nomis TS066,2021,Persons,All usual residents aged 16+ in the economic a...,Not applicable — count,Retained source count
8,LSOA_Code,object,Lower-layer Super Output Area code used as the...,ONS Census 2021 geography via Nomis,2021,Text identifier,Included LSOAs,Not applicable,Retained from cleaned Census data
9,LSOA_Name,object,Census 2021 Lower-layer Super Output Area name.,ONS Census 2021 geography via Nomis,2021,Text identifier,Included LSOAs,Not applicable,Retained from cleaned Census data


### Findings — Census dictionary coverage

The Census dictionary contains **50 entries**, covering every Census variable and geographic identifier in the prepared dataset. Coverage checks found no missing entries, unexpected entries or duplicate variable names.

Each entry records its meaning, data type, source, reference year, units, population coverage, denominator and calculation. Counts and percentages are distinguished, and each percentage is linked to its source-specific denominator.

These checks confirm dictionary coverage, not independent verification of the underlying source values. The remaining **55 IoD2019 columns** still need to be documented before the full dictionary is exported.

### 14.2 Historical deprivation variables

Document the 55 IoD2019 columns, including administrative identifiers, deprivation measures and supporting population counts.

The release year is 2019, but underlying observation dates vary. Scores, rates, ranks and deciles have different meanings and units. Published deprivation measures are retained as supplied rather than recalculated using Census 2021 denominators.

Ranks and deciles describe relative deprivation across England, not rankings within the 199 included Dudley neighbourhoods.

In [19]:
# Developed with ChatGPT assistance.
iod_rows = []


def add_iod(
    column,
    meaning,
    units,
    population,
    denominator,
    year,
    calculation
):
    iod_rows.append({
        "Variable": column,
        "Data_Type": str(dictionary_data[column].dtype),
        "Meaning": meaning,
        "Source": (
            "MHCLG English Indices of Deprivation 2019"
        ),
        "Reference_Year": year,
        "Units": units,
        "Population_Coverage": population,
        "Denominator": denominator,
        "Calculation": calculation
    })


# Describe each index, domain or sub-domain once.
domain_meanings = {
    "IMD": "Overall multiple deprivation",
    "Income": "Income deprivation",
    "Employment": (
        "Involuntary exclusion from employment"
    ),
    "Education_Skills_and_Training": (
        "Deprivation in education and skills"
    ),
    "Health_Deprivation_and_Disability": (
        "Poor health, disability "
        "and premature mortality"
    ),
    "Crime": (
        "Risk of personal and material victimisation"
    ),
    "Barriers_to_Housing_and_Services": (
        "Barriers to housing and local services"
    ),
    "Living_Environment": (
        "Deprivation in indoor "
        "and outdoor living conditions"
    ),
    "IDACI": "Income deprivation affecting children",
    "IDAOPI": (
        "Income deprivation affecting older people"
    ),
    "Children_and_Young_People_Sub_domain": (
        "Educational deprivation among "
        "children and young people"
    ),
    "Adult_Skills_Sub_domain": (
        "Adult qualification and "
        "English-language skill deficits"
    ),
    "Geographical_Barriers_Sub_domain": (
        "Distance to selected local services"
    ),
    "Wider_Barriers_Sub_domain": (
        "Housing affordability, overcrowding "
        "and homelessness"
    ),
    "Indoors_Sub_domain": (
        "Housing condition and "
        "central-heating deprivation"
    ),
    "Outdoors_Sub_domain": (
        "Air quality and "
        "road-traffic-accident deprivation"
    )
}

# Retained rates, not calculated percentages.
rate_populations = {
    "Income": (
        "Population excluding prisoners, "
        "using IoD source denominators"
    ),
    "Employment": (
        "Women aged 18–59 and men aged 18–64, "
        "excluding prisoners; "
        "IoD employment denominator uses "
        "2015/2016 population estimates"
    ),
    "IDACI": (
        "Children aged 0–15, excluding prisoners"
    ),
    "IDAOPI": (
        "People aged 60+, excluding prisoners"
    )
}

# Document authority identifiers separately.
authority_definitions = {
    "code": (
        "Local authority district code "
        "using the 2019 classification"
    ),
    "name": (
        "Local authority district name "
        "using the 2019 classification"
    )
}

for ending, meaning in authority_definitions.items():
    add_iod(
        (
            "IoD2019_Local_Authority_District_"
            f"{ending}_2019"
        ),
        meaning,
        "Text identifier",
        "Local authority containing the LSOA",
        "Not applicable",
        "2019 administrative classification",
        "Retained source identifier"
    )

# Create score, rank and decile entries.
for domain, meaning in domain_meanings.items():
    is_rate = domain in rate_populations
    score_suffix = "Score_rate" if is_rate else "Score"
    score_column = f"IoD2019_{domain}_{score_suffix}"

    add_iod(
        score_column,
        (
            f"{meaning}; higher values "
            "indicate greater deprivation."
        ),
        (
            "Proportion (0–1)"
            if is_rate
            else "Composite score; not a percentage"
        ),
        rate_populations.get(
            domain,
            (
                "2011-boundary LSOAs; "
                "component populations vary"
            )
        ),
        rate_populations.get(
            domain,
            "No single denominator — composite measure"
        ),
        "2019 release; underlying dates vary",
        (
            "Published measure retained; "
            "not recalculated from Census 2021"
        )
    )

    add_iod(
        f"IoD2019_{domain}_Rank",
        (
            f"National rank for {meaning.lower()}; "
            "1 is most deprived."
        ),
        "Ordinal rank (1–32,844)",
        (
            "32,844 LSOAs in England "
            "using 2011 boundaries"
        ),
        "Not applicable — national ordering",
        "2019 release; underlying dates vary",
        "Retained published national rank"
    )

    add_iod(
        f"IoD2019_{domain}_Decile",
        (
            f"National decile for {meaning.lower()}; "
            "1 is most deprived and 10 least deprived."
        ),
        "Ordinal group (1–10)",
        "LSOAs in England using 2011 boundaries",
        "Not applicable — national grouping",
        "2019 release; underlying dates vary",
        "Retained published national decile"
    )

# Record supporting population counts.
population_definitions = {
    (
        "Total_population_mid_2015_"
        "excluding_prisoners"
    ): (
        "Total population excluding prisoners",
        "Mid-2015"
    ),
    (
        "Dependent_Children_aged_0_15_"
        "mid_2015_excluding_prisoners"
    ): (
        (
            "Dependent children aged 0–15 "
            "excluding prisoners"
        ),
        "Mid-2015"
    ),
    (
        "Population_aged_16_59_mid_2015_"
        "excluding_prisoners"
    ): (
        "Population aged 16–59 excluding prisoners",
        "Mid-2015"
    ),
    (
        "Older_population_aged_60_and_over_"
        "mid_2015_excluding_prisoners"
    ): (
        "Population aged 60+ excluding prisoners",
        "Mid-2015"
    ),
    (
        "Working_age_population_18_59_64_"
        "for_use_with_Employment_Deprivation_"
        "Domain_excluding_prisoners"
    ): (
        (
            "Employment-domain population: "
            "women aged 18–59 and men aged 18–64, "
            "excluding prisoners"
        ),
        (
            "2015/2016 population estimates used "
            "in IoD2019 employment methodology"
        )
    )
}

for suffix, (meaning, year) in (
    population_definitions.items()
):
    add_iod(
        f"IoD2019_{suffix}",
        meaning,
        "Persons",
        meaning,
        "Not applicable — count",
        year,
        "Retained supporting population count"
    )

# Check dictionary coverage.
iod_dictionary = pd.DataFrame(iod_rows)

expected_iod = [
    column
    for column in dictionary_data.columns
    if column.startswith("IoD2019_")
]

missing_entries = sorted(
    set(expected_iod)
    - set(iod_dictionary["Variable"])
)

unexpected_entries = sorted(
    set(iod_dictionary["Variable"])
    - set(expected_iod)
)

duplicate_entries = (
    iod_dictionary["Variable"].duplicated().sum()
)

print("IoD2019 entries:", len(iod_dictionary))
print("Missing entries:", missing_entries)
print("Unexpected entries:", unexpected_entries)
print("Duplicate entries:", duplicate_entries)

if (
    missing_entries
    or unexpected_entries
    or duplicate_entries
):
    raise ValueError(
        "Review IoD dictionary coverage "
        "before continuing."
    )

# Combine and match the original CSV column order.
data_dictionary = pd.concat(
    [census_dictionary, iod_dictionary],
    ignore_index=True
)

data_dictionary = (
    data_dictionary.set_index("Variable")
    .loc[dictionary_data.columns]
    .reset_index()
)

print(
    "Total dictionary entries:",
    len(data_dictionary)
)
display(iod_dictionary.head(8))

IoD2019 entries: 55
Missing entries: []
Unexpected entries: []
Duplicate entries: 0
Total dictionary entries: 105


,Variable,Data_Type,Meaning,Source,Reference_Year,Units,Population_Coverage,Denominator,Calculation
0,IoD2019_Local_Authority_District_code_2019,object,Local authority district code using the 2019 c...,MHCLG English Indices of Deprivation 2019,2019 administrative classification,Text identifier,Local authority containing the LSOA,Not applicable,Retained source identifier
1,IoD2019_Local_Authority_District_name_2019,object,Local authority district name using the 2019 c...,MHCLG English Indices of Deprivation 2019,2019 administrative classification,Text identifier,Local authority containing the LSOA,Not applicable,Retained source identifier
2,IoD2019_IMD_Score,float64,Overall multiple deprivation; higher values in...,MHCLG English Indices of Deprivation 2019,2019 release; underlying dates vary,Composite score; not a percentage,2011-boundary LSOAs; component populations vary,No single denominator — composite measure,Published measure retained; not recalculated f...
3,IoD2019_IMD_Rank,int64,National rank for overall multiple deprivation...,MHCLG English Indices of Deprivation 2019,2019 release; underlying dates vary,"Ordinal rank (1–32,844)","32,844 LSOAs in England using 2011 boundaries",Not applicable — national ordering,Retained published national rank
4,IoD2019_IMD_Decile,int64,National decile for overall multiple deprivati...,MHCLG English Indices of Deprivation 2019,2019 release; underlying dates vary,Ordinal group (1–10),LSOAs in England using 2011 boundaries,Not applicable — national grouping,Retained published national decile
5,IoD2019_Income_Score_rate,float64,Income deprivation; higher values indicate gre...,MHCLG English Indices of Deprivation 2019,2019 release; underlying dates vary,Proportion (0–1),"Population excluding prisoners, using IoD sour...","Population excluding prisoners, using IoD sour...",Published measure retained; not recalculated f...
6,IoD2019_Income_Rank,int64,National rank for income deprivation; 1 is mos...,MHCLG English Indices of Deprivation 2019,2019 release; underlying dates vary,"Ordinal rank (1–32,844)","32,844 LSOAs in England using 2011 boundaries",Not applicable — national ordering,Retained published national rank
7,IoD2019_Income_Decile,int64,National decile for income deprivation; 1 is m...,MHCLG English Indices of Deprivation 2019,2019 release; underlying dates vary,Ordinal group (1–10),LSOAs in England using 2011 boundaries,Not applicable — national grouping,Retained published national decile


### Findings — Complete dictionary coverage

The IoD2019 dictionary contains **55 entries**, with no missing, unexpected or duplicate variable names. Combined with the 50 Census and geographic entries, the dictionary covers all **105 columns** in the prepared dataset.

The entries distinguish population counts, proportions, composite scores, national ranks and deciles. They also distinguish the 2019 release from underlying observation dates and identify the relevant population coverage and denominators.

The combined dictionary follows the prepared CSV’s column order. These checks confirm complete column coverage; they do not independently verify the source values. The two selected-inactivity variables created separately in notebook 08 still require supplementary documentation.

### 14.3 Supplementary hypothesis-testing variables

Document the two selected-inactivity variables created in notebook 08. These are analysis variables and are not included in the 105-column prepared dataset, so their definitions are kept in a separate supplementary dictionary.

The numerator combines economic inactivity due to looking after home or family, long-term sickness or disability, and other reasons. It excludes retirement and inactive students. The denominator includes all usual residents aged 16 years and over, so the resulting percentage must not be described as total economic inactivity or working-age inactivity.

These area-level measures describe selected Census categories, not individual circumstances or willingness to work. Associations with structural barriers do not establish causation.

In [20]:
# Code developed with ChatGPT assistance.
# Document analysis variables separately from prepared-data columns.
supplementary_rows = [
    {
        "Variable": "Selected_Inactive_Count",
        "Data_Type": "Integer-valued count",
        "Meaning": (
            "Residents in the selected home/family, long-term "
            "sickness/disability and other inactivity categories"
        ),
        "Source": "ONS Census 2021, TS066, accessed via Nomis",
        "Reference_Year": "2021",
        "Units": "Persons",
        "Population_Coverage": (
            "Usual residents aged 16 years and over "
            "in the 199 included Dudley LSOAs"
        ),
        "Denominator": "Not applicable — count",
        "Calculation": (
            "Inactive_Home_Or_Family + "
            "Inactive_Long_Term_Sick_Or_Disabled + "
            "Inactive_Other"
        ),
    },
    {
        "Variable": "Selected_Inactive_Pct",
        "Data_Type": "Floating-point percentage",
        "Meaning": (
            "Selected inactive residents as a percentage "
            "of all usual residents aged 16 years and over"
        ),
        "Source": "ONS Census 2021, TS066, accessed via Nomis",
        "Reference_Year": "2021",
        "Units": "Percentage (0–100)",
        "Population_Coverage": (
            "Usual residents aged 16 years and over "
            "in the 199 included Dudley LSOAs"
        ),
        "Denominator": "Economic_Total_16_Plus",
        "Calculation": (
            "Selected_Inactive_Count / "
            "Economic_Total_16_Plus * 100"
        ),
    },
]

supplementary_dictionary = pd.DataFrame(supplementary_rows)

# Identify the notebook where these variables are calculated.
supplementary_dictionary["Created_In"] = (
    "jupyter_notebooks/08_Hypothesis_Testing.ipynb"
)

print(
    "Prepared-data dictionary entries:",
    len(data_dictionary),
)
print(
    "Supplementary entries:",
    len(supplementary_dictionary),
)

display(supplementary_dictionary)

Prepared-data dictionary entries: 105
Supplementary entries: 2


,Variable,Data_Type,Meaning,Source,Reference_Year,Units,Population_Coverage,Denominator,Calculation,Created_In
0,Selected_Inactive_Count,Integer-valued count,"Residents in the selected home/family, long-te...","ONS Census 2021, TS066, accessed via Nomis",2021,Persons,Usual residents aged 16 years and over in the ...,Not applicable — count,Inactive_Home_Or_Family + Inactive_Long_Term_S...,jupyter_notebooks/08_Hypothesis_Testing.ipynb
1,Selected_Inactive_Pct,Floating-point percentage,Selected inactive residents as a percentage of...,"ONS Census 2021, TS066, accessed via Nomis",2021,Percentage (0–100),Usual residents aged 16 years and over in the ...,Economic_Total_16_Plus,Selected_Inactive_Count / Economic_Total_16_Pl...,jupyter_notebooks/08_Hypothesis_Testing.ipynb


#### Findings — Supplementary variable documentation

The supplementary dictionary documents `Selected_Inactive_Count` and `Selected_Inactive_Pct`, including their source, population coverage, units and calculations. Both variables are created in notebook 08 and remain separate from the 105 columns in the prepared dataset.

The percentage uses all usual residents aged 16 years and over as its denominator. Its numerator includes only home or family responsibilities, long-term sickness or disability, and other inactivity reasons. It therefore represents selected economic inactivity categories, not total or working-age economic inactivity.

This step documents the definitions; it does not independently verify the calculations against the original source data. Neither dictionary has been exported yet.

### 14.4 Export the data dictionaries

Save the dictionary for the 105 prepared-data columns and the supplementary dictionary for the two variables created in notebook 08 as separate CSV files. This preserves the distinction between the saved prepared dataset and the variables derived during hypothesis testing.

The exports provide reusable documentation of variable meanings, sources, population coverage, units, denominators and calculations.

In [21]:
# Code developed with ChatGPT assistance.
from pathlib import Path

# Reference: Python pathlib documentation.
# https://docs.python.org/3/library/pathlib.html
# Used to check cwd(), parent and mkdir() for managing output paths.

# Locate the project from its root or the notebooks folder.
project_root = Path.cwd()
if project_root.name == "jupyter_notebooks":
    project_root = project_root.parent

# Create missing parent folders and allow an existing output folder.
dictionary_dir = project_root / "outputs" / "data_dictionary"
dictionary_dir.mkdir(parents=True, exist_ok=True)

prepared_path = dictionary_dir / "prepared_data_dictionary.csv"
supplementary_path = (
    dictionary_dir / "supplementary_data_dictionary.csv"
)

# Reference: pandas DataFrame.to_csv documentation.
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html
# Used to check that index=False excludes row numbers from the CSV.

# Save separate files to distinguish prepared and derived variables.
data_dictionary.to_csv(prepared_path, index=False)
supplementary_dictionary.to_csv(supplementary_path, index=False)

# Display project-relative paths and the number of exported entries.
print("Saved:", prepared_path.relative_to(project_root))
print("Prepared-data entries:", len(data_dictionary))
print("Saved:", supplementary_path.relative_to(project_root))
print("Supplementary entries:", len(supplementary_dictionary))

Saved: outputs\data_dictionary\prepared_data_dictionary.csv
Prepared-data entries: 105
Saved: outputs\data_dictionary\supplementary_data_dictionary.csv
Supplementary entries: 2


#### Findings — Data dictionary exports

The prepared-data dictionary was exported with 105 entries, and the supplementary dictionary was exported separately with two entries. Both files are saved in `outputs/data_dictionary/`.

The separate exports distinguish the prepared dataset’s columns from the selected-inactivity variables created in notebook 08. Source comparisons remain outstanding before the validation task is complete.

---

### 15. Verify selected values against the raw sources

Inspect the original source files before comparing selected counts and recalculated percentages with the prepared dataset. These comparisons will check whether selected source values were preserved during cleaning and merging.

#### 15.1 Inspect source headers

Display the opening lines of the five raw datasets to identify their headers, metadata and area-code fields.

In [22]:
# Code developed with ChatGPT assistance.
from pathlib import Path

# Reference: Python pathlib documentation.
# https://docs.python.org/3/library/pathlib.html
# Used for locating files and opening them through Path objects.

# Locate the project from its root or the notebooks folder.
project_root = Path.cwd()
if project_root.name == "jupyter_notebooks":
    project_root = project_root.parent

raw_dir = project_root / "raw_datasets"

source_files = [
    "dudley_economic_activity.csv",
    "dudley_qualifications.csv",
    "dudley_disability_16_plus.csv",
    "dudley_car_availability.csv",
    "dudley_iod.csv",
]

# Inspect metadata and headers before choosing how to read each file.
# utf-8-sig handles an optional byte-order mark at the file's start.
for filename in source_files:
    source_path = raw_dir / filename
    print(f"\nFILE: {filename}")

    with source_path.open(encoding="utf-8-sig") as source:
        for line_number in range(1, 11):
            line = source.readline()
            if not line:
                break
            print(f"{line_number}: {line.rstrip()}")


FILE: dudley_economic_activity.csv
1: 
2: "TS066 - Economic activity status"
3: "ONS Crown Copyright Reserved [from Nomis on 14 October 2026]"
4: "Population :","All usual residents aged 16 years and over"
5: "Units      :","Persons"
6: "Date       :","2021"
7: 
8: "2021 super output area - lower layer","Total: All usual residents aged 16 years and over","Economically active (excluding full-time students)","Economically active (excluding full-time students):In employment","Economically active (excluding full-time students): Unemployed","Economically inactive: Student","Economically inactive: Looking after home or family","Economically inactive: Long-term sick or disabled","Economically inactive: Other"
9: 
10: "E01009763 : Dudley 001A",1637,970,888,82,97,164,73,70

FILE: dudley_qualifications.csv
1: 
2: "TS067 - Highest level of qualification"
3: "ONS Crown Copyright Reserved [from Nomis on 14 October 2026]"
4: "Population :","All usual residents aged 16 years and over"
5: "Units     

#### Findings — Raw source structure

The economic activity, qualifications and disability files have their column headers on line 8. The car availability file has its header on line 7, while the deprivation file starts with its header on line 1. Metadata and empty separator rows must be handled when loading the Census files.

The economic activity file combines the LSOA code and name in one field. Qualifications, disability and car availability provide a separate `mnemonic` code field. The deprivation file uses `LSOA code (2011)` and contains areas outside Dudley, so comparisons must select the included LSOA codes.

For Dudley 001A, the economic activity and qualifications sources each report 1,637 residents aged 16 and over, while the disability source reports 1,640. The car availability source reports 882 households. Each percentage must therefore retain its own source-specific denominator; these totals must not be substituted for one another.

This inspection establishes how to read and match the sources. Comparisons with the prepared dataset have not yet been performed.

#### 15.2 Verify qualification values against the raw source

Compare the qualification population totals and no-qualification counts in the prepared dataset with the original TS067 file for every included LSOA. Independently recalculate the no-qualification percentage from the raw counts.

Counts must match exactly. Percentages are compared with an absolute tolerance of 0.00000001 percentage points to allow for floating-point precision. Unmatched areas and discrepancies are reported rather than corrected automatically.

In [23]:
# Code developed with ChatGPT assistance.
import numpy as np
import pandas as pd

# Reference: pandas read_csv documentation.
# https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html
# Used to check skiprows for loading the header after metadata.
raw_qual = pd.read_csv(
    raw_dir / "dudley_qualifications.csv",
    skiprows=7,
    encoding="utf-8-sig",
)

# Keep area records and select the original counts for comparison.
raw_qual = raw_qual.loc[
    raw_qual["mnemonic"].str.fullmatch(r"E\d{8}", na=False)
].copy()

raw_total = "Total: All usual residents aged 16 years and over"
raw_qual = raw_qual[
    ["mnemonic", raw_total, "No qualifications"]
].rename(
    columns={
        "mnemonic": "LSOA_Code",
        raw_total: "Raw_Total",
        "No qualifications": "Raw_No_Qualifications",
    }
)

for column in ["Raw_Total", "Raw_No_Qualifications"]:
    raw_qual[column] = pd.to_numeric(
        raw_qual[column], errors="raise"
    )

# Read the saved prepared file so checks cover the exported data.
prepared_file = (
    project_root / "cleaned_datasets"
    / "dudley_combined_prepared.csv"
)
prepared_check = pd.read_csv(prepared_file)

qual_columns = [
    "LSOA_Code",
    "Qualifications_Total_16_Plus",
    "No_Qualifications",
    "No_Qualifications_Pct",
]

# Retain every prepared area and reject duplicate matching keys.
qual_check = prepared_check[qual_columns].merge(
    raw_qual,
    on="LSOA_Code",
    how="left",
    validate="one_to_one",
    indicator=True,
)

# Recalculate percentages using only the original source counts.
qual_check["Raw_Percentage"] = (
    qual_check["Raw_No_Qualifications"]
    / qual_check["Raw_Total"]
    * 100
)

qual_check["Total_Matches"] = (
    qual_check["Qualifications_Total_16_Plus"]
    == qual_check["Raw_Total"]
)
qual_check["Count_Matches"] = (
    qual_check["No_Qualifications"]
    == qual_check["Raw_No_Qualifications"]
)

# Reference: NumPy isclose documentation.
# https://numpy.org/doc/stable/reference/generated/numpy.isclose.html
# Used to set an explicit tolerance for floating-point comparisons.
qual_check["Percentage_Matches"] = np.isclose(
    qual_check["No_Qualifications_Pct"],
    qual_check["Raw_Percentage"],
    rtol=0,
    atol=1e-8,
)

check_columns = [
    "Total_Matches",
    "Count_Matches",
    "Percentage_Matches",
]
qual_issues = qual_check.loc[
    ~qual_check[check_columns].all(axis=1)
]

print("Areas checked:", len(qual_check))
print("Unmatched areas:", (qual_check["_merge"] != "both").sum())
print("Total mismatches:", (~qual_check["Total_Matches"]).sum())
print("Count mismatches:", (~qual_check["Count_Matches"]).sum())
print(
    "Percentage mismatches:",
    (~qual_check["Percentage_Matches"]).sum(),
)

display(qual_check.head())
display(qual_issues)

Areas checked: 199
Unmatched areas: 0
Total mismatches: 0
Count mismatches: 0
Percentage mismatches: 0


,LSOA_Code,Qualifications_Total_16_Plus,No_Qualifications,No_Qualifications_Pct,Raw_Total,Raw_No_Qualifications,_merge,Raw_Percentage,Total_Matches,Count_Matches,Percentage_Matches
0,E01009763,1637,436,26.634087,1637.0,436.0,both,26.634087,True,True,True
1,E01009764,1266,306,24.170616,1266.0,306.0,both,24.170616,True,True,True
2,E01009767,1265,358,28.300395,1265.0,358.0,both,28.300395,True,True,True
3,E01009769,1428,467,32.703081,1428.0,467.0,both,32.703081,True,True,True
4,E01009770,1215,390,32.098765,1215.0,390.0,both,32.098765,True,True,True


,LSOA_Code,Qualifications_Total_16_Plus,No_Qualifications,No_Qualifications_Pct,Raw_Total,Raw_No_Qualifications,_merge,Raw_Percentage,Total_Matches,Count_Matches,Percentage_Matches


#### Findings — Qualification source verification

All 199 included LSOAs matched the original TS067 qualifications file. The prepared qualification population totals and no-qualification counts matched the raw source exactly for every area.

The no-qualification percentages recalculated directly from the raw counts matched the saved percentages within the specified tolerance of 0.00000001 percentage points. No discrepancies were identified, so no corrections were required.

These checks verify the selected qualification fields across all included areas. They do not yet verify the other source datasets.

#### 15.3 Verify disability values against the raw source

Compare the disability population totals and counts of residents whose day-to-day activities are limited a lot with the original RM019 file for every included LSOA. Recalculate the percentage using the disability source’s own population total.

Counts must match exactly. Percentages use an absolute tolerance of 0.00000001 percentage points. Report unmatched areas and discrepancies without changing the prepared dataset.

In [24]:
# Code developed with ChatGPT assistance.

# Reference: pandas read_csv documentation.
# https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html
# Used to check skiprows for loading the header after metadata.
raw_disability = pd.read_csv(
    raw_dir / "dudley_disability_16_plus.csv",
    skiprows=7,
    encoding="utf-8-sig",
)

# Keep area records and select the original source counts.
raw_disability = raw_disability.loc[
    raw_disability["mnemonic"].str.fullmatch(r"E\d{8}", na=False)
].copy()

limited_column = (
    "Disabled under the Equality Act: "
    "Day-to-day activities limited a lot"
)

raw_disability = raw_disability[
    ["mnemonic", "Total", limited_column]
].rename(
    columns={
        "mnemonic": "LSOA_Code",
        "Total": "Raw_Total",
        limited_column: "Raw_Limited_A_Lot",
    }
)

for column in ["Raw_Total", "Raw_Limited_A_Lot"]:
    raw_disability[column] = pd.to_numeric(
        raw_disability[column], errors="raise"
    )

disability_columns = [
    "LSOA_Code",
    "Disability_Total_16_Plus",
    "Limited_A_Lot_16_Plus",
    "Limited_A_Lot_16_Plus_Pct",
]

# Retain every prepared area and reject duplicate matching keys.
disability_check = prepared_check[disability_columns].merge(
    raw_disability,
    on="LSOA_Code",
    how="left",
    validate="one_to_one",
    indicator=True,
)

# Use the disability denominator, not the qualifications total.
disability_check["Raw_Percentage"] = (
    disability_check["Raw_Limited_A_Lot"]
    / disability_check["Raw_Total"]
    * 100
)

disability_check["Total_Matches"] = (
    disability_check["Disability_Total_16_Plus"]
    == disability_check["Raw_Total"]
)
disability_check["Count_Matches"] = (
    disability_check["Limited_A_Lot_16_Plus"]
    == disability_check["Raw_Limited_A_Lot"]
)

# Reference: NumPy isclose documentation.
# https://numpy.org/doc/stable/reference/generated/numpy.isclose.html
# Used to set an explicit tolerance for floating-point comparisons.
disability_check["Percentage_Matches"] = np.isclose(
    disability_check["Limited_A_Lot_16_Plus_Pct"],
    disability_check["Raw_Percentage"],
    rtol=0,
    atol=1e-8,
)

disability_checks = [
    "Total_Matches",
    "Count_Matches",
    "Percentage_Matches",
]
disability_issues = disability_check.loc[
    ~disability_check[disability_checks].all(axis=1)
]

print("Areas checked:", len(disability_check))
print(
    "Unmatched areas:",
    (disability_check["_merge"] != "both").sum(),
)
print(
    "Total mismatches:",
    (~disability_check["Total_Matches"]).sum(),
)
print(
    "Count mismatches:",
    (~disability_check["Count_Matches"]).sum(),
)
print(
    "Percentage mismatches:",
    (~disability_check["Percentage_Matches"]).sum(),
)

display(disability_check.head())
display(disability_issues)

Areas checked: 199
Unmatched areas: 0
Total mismatches: 0
Count mismatches: 0
Percentage mismatches: 0


,LSOA_Code,Disability_Total_16_Plus,Limited_A_Lot_16_Plus,Limited_A_Lot_16_Plus_Pct,Raw_Total,Raw_Limited_A_Lot,_merge,Raw_Percentage,Total_Matches,Count_Matches,Percentage_Matches
0,E01009763,1640,154,9.390244,1640.0,154.0,both,9.390244,True,True,True
1,E01009764,1264,121,9.572785,1264.0,121.0,both,9.572785,True,True,True
2,E01009767,1268,145,11.435331,1268.0,145.0,both,11.435331,True,True,True
3,E01009769,1428,208,14.565826,1428.0,208.0,both,14.565826,True,True,True
4,E01009770,1213,136,11.211871,1213.0,136.0,both,11.211871,True,True,True


,LSOA_Code,Disability_Total_16_Plus,Limited_A_Lot_16_Plus,Limited_A_Lot_16_Plus_Pct,Raw_Total,Raw_Limited_A_Lot,_merge,Raw_Percentage,Total_Matches,Count_Matches,Percentage_Matches


#### Findings — Disability source verification

All 199 included LSOAs matched the original RM019 disability file. The prepared disability population totals and counts of residents whose day-to-day activities are limited a lot matched the raw source exactly.

Percentages recalculated from the raw disability counts and their source-specific population totals matched the saved percentages within the specified tolerance of 0.00000001 percentage points. No discrepancies were identified, so no corrections were required.

These checks confirm that the selected disability fields preserve the original source values and use the disability denominator rather than a total from another Census table.

#### 15.4 Verify car availability values against the raw source

Compare household totals and counts of households without a car or van with the original TS045 file for every included LSOA. Independently recalculate the no-car percentage using the raw household totals.

Counts must match exactly. Percentages use an absolute tolerance of 0.00000001 percentage points. This checks the calculation and source agreement; it does not establish that households without a car experience transport poverty.

In [25]:
# Code developed with ChatGPT assistance.

# Reference: pandas read_csv documentation.
# https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html
# Used to check skiprows for loading the header after metadata.
raw_car = pd.read_csv(
    raw_dir / "dudley_car_availability.csv",
    skiprows=6,
    encoding="utf-8-sig",
)

# Keep area records and select the original household counts.
raw_car = raw_car.loc[
    raw_car["mnemonic"].str.fullmatch(r"E\d{8}", na=False)
].copy()

raw_car = raw_car[
    [
        "mnemonic",
        "Total: All households",
        "No cars or vans in household",
    ]
].rename(
    columns={
        "mnemonic": "LSOA_Code",
        "Total: All households": "Raw_Total",
        "No cars or vans in household": "Raw_No_Car",
    }
)

for column in ["Raw_Total", "Raw_No_Car"]:
    raw_car[column] = pd.to_numeric(
        raw_car[column], errors="raise"
    )

car_columns = [
    "LSOA_Code",
    "Total_Households",
    "No_Car_Van_Households",
    "No_Car_Van_Households_Pct",
]

# Retain every prepared area and reject duplicate matching keys.
car_check = prepared_check[car_columns].merge(
    raw_car,
    on="LSOA_Code",
    how="left",
    validate="one_to_one",
    indicator=True,
)

# Recalculate using households, not the resident population.
car_check["Raw_Percentage"] = (
    car_check["Raw_No_Car"]
    / car_check["Raw_Total"]
    * 100
)

car_check["Total_Matches"] = (
    car_check["Total_Households"] == car_check["Raw_Total"]
)
car_check["Count_Matches"] = (
    car_check["No_Car_Van_Households"] == car_check["Raw_No_Car"]
)

# Reference: NumPy isclose documentation.
# https://numpy.org/doc/stable/reference/generated/numpy.isclose.html
# Used to set an explicit tolerance for floating-point comparisons.
car_check["Percentage_Matches"] = np.isclose(
    car_check["No_Car_Van_Households_Pct"],
    car_check["Raw_Percentage"],
    rtol=0,
    atol=1e-8,
)

car_checks = [
    "Total_Matches",
    "Count_Matches",
    "Percentage_Matches",
]
car_issues = car_check.loc[
    ~car_check[car_checks].all(axis=1)
]

print("Areas checked:", len(car_check))
print("Unmatched areas:", (car_check["_merge"] != "both").sum())
print("Total mismatches:", (~car_check["Total_Matches"]).sum())
print("Count mismatches:", (~car_check["Count_Matches"]).sum())
print(
    "Percentage mismatches:",
    (~car_check["Percentage_Matches"]).sum(),
)

display(car_check.head())
display(car_issues)

Areas checked: 199
Unmatched areas: 0
Total mismatches: 0
Count mismatches: 0
Percentage mismatches: 0


,LSOA_Code,Total_Households,No_Car_Van_Households,No_Car_Van_Households_Pct,Raw_Total,Raw_No_Car,_merge,Raw_Percentage,Total_Matches,Count_Matches,Percentage_Matches
0,E01009763,882,261,29.591837,882.0,261.0,both,29.591837,True,True,True
1,E01009764,632,76,12.025316,632.0,76.0,both,12.025316,True,True,True
2,E01009767,658,152,23.100304,658.0,152.0,both,23.100304,True,True,True
3,E01009769,824,336,40.776699,824.0,336.0,both,40.776699,True,True,True
4,E01009770,637,152,23.861852,637.0,152.0,both,23.861852,True,True,True


,LSOA_Code,Total_Households,No_Car_Van_Households,No_Car_Van_Households_Pct,Raw_Total,Raw_No_Car,_merge,Raw_Percentage,Total_Matches,Count_Matches,Percentage_Matches


#### Findings — Car availability source verification

All 199 included LSOAs matched the original TS045 car availability file. The prepared household totals and counts of households without a car or van matched the raw source exactly.

Percentages recalculated from the raw household counts matched the saved percentages within the specified tolerance of 0.00000001 percentage points. No discrepancies were identified, so no corrections were required.

These checks confirm source agreement and the use of household denominators. They do not establish that households without a car experience transport poverty, because car availability alone does not measure transport accessibility or affordability.

#### 15.5 Verify selected economic activity values against the raw source

Compare the population totals and the home or family, long-term sickness or disability, and other inactivity counts with the original TS066 file for every included LSOA. Recalculate each selected category’s percentage using the raw economic activity population total.

These are the source categories used to construct the selected-inactivity outcome in notebook 08. Counts must match exactly, while percentages use an absolute tolerance of 0.00000001 percentage points. This check does not redefine these categories as total economic inactivity.

In [26]:
# Code developed with ChatGPT assistance.

# Reference: pandas read_csv documentation.
# https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html
# Used to check skiprows for loading the header after metadata.
raw_economic = pd.read_csv(
    raw_dir / "dudley_economic_activity.csv",
    skiprows=7,
    encoding="utf-8-sig",
)

# Extract the area code from the combined code-and-name field.
area_column = "2021 super output area - lower layer"
raw_economic["LSOA_Code"] = (
    raw_economic[area_column].str.extract(r"^(E\d{8})\s*:")
)
raw_economic = raw_economic.loc[
    raw_economic["LSOA_Code"].notna()
].copy()

# Map the original headings to the prepared column names.
economic_mapping = {
    "Total: All usual residents aged 16 years and over": (
        "Economic_Total_16_Plus"
    ),
    "Economically inactive: Looking after home or family": (
        "Inactive_Home_Or_Family"
    ),
    "Economically inactive: Long-term sick or disabled": (
        "Inactive_Long_Term_Sick_Or_Disabled"
    ),
    "Economically inactive: Other": "Inactive_Other",
}

economic_counts = list(economic_mapping.values())
economic_categories = economic_counts[1:]
economic_percentages = [
    f"{column}_Pct" for column in economic_categories
]

raw_economic = raw_economic[
    ["LSOA_Code"] + list(economic_mapping)
].rename(columns=economic_mapping)

for column in economic_counts:
    raw_economic[column] = pd.to_numeric(
        raw_economic[column], errors="raise"
    )

# Prefix raw counts to keep both versions visible after matching.
raw_economic = raw_economic.rename(
    columns={column: f"Raw_{column}" for column in economic_counts}
)

economic_columns = (
    ["LSOA_Code"] + economic_counts + economic_percentages
)

# Retain every prepared area and reject duplicate matching keys.
economic_check = prepared_check[economic_columns].merge(
    raw_economic,
    on="LSOA_Code",
    how="left",
    validate="one_to_one",
    indicator=True,
)

# Check the population total and all three counts exactly.
economic_checks = []
for column in economic_counts:
    match_column = f"{column}_Matches"
    economic_check[match_column] = (
        economic_check[column] == economic_check[f"Raw_{column}"]
    )
    economic_checks.append(match_column)

# Reference: NumPy isclose documentation.
# https://numpy.org/doc/stable/reference/generated/numpy.isclose.html
# Used to set an explicit tolerance for percentage comparisons.
for column in economic_categories:
    raw_percentage = (
        economic_check[f"Raw_{column}"]
        / economic_check["Raw_Economic_Total_16_Plus"]
        * 100
    )
    economic_check[f"Raw_{column}_Pct"] = raw_percentage
    match_column = f"{column}_Pct_Matches"
    economic_check[match_column] = np.isclose(
        economic_check[f"{column}_Pct"],
        raw_percentage,
        rtol=0,
        atol=1e-8,
    )
    economic_checks.append(match_column)

economic_issues = economic_check.loc[
    ~economic_check[economic_checks].all(axis=1)
]

print("Areas checked:", len(economic_check))
print(
    "Unmatched areas:",
    (economic_check["_merge"] != "both").sum(),
)

for column in economic_checks:
    print(column, "— mismatches:", (~economic_check[column]).sum())

display(economic_check.head())
display(economic_issues)

Areas checked: 199
Unmatched areas: 0
Economic_Total_16_Plus_Matches — mismatches: 0
Inactive_Home_Or_Family_Matches — mismatches: 0
Inactive_Long_Term_Sick_Or_Disabled_Matches — mismatches: 0
Inactive_Other_Matches — mismatches: 0
Inactive_Home_Or_Family_Pct_Matches — mismatches: 0
Inactive_Long_Term_Sick_Or_Disabled_Pct_Matches — mismatches: 0
Inactive_Other_Pct_Matches — mismatches: 0


,LSOA_Code,Economic_Total_16_Plus,Inactive_Home_Or_Family,Inactive_Long_Term_Sick_Or_Disabled,Inactive_Other,Inactive_Home_Or_Family_Pct,Inactive_Long_Term_Sick_Or_Disabled_Pct,Inactive_Other_Pct,Raw_Economic_Total_16_Plus,Raw_Inactive_Home_Or_Family,...,Economic_Total_16_Plus_Matches,Inactive_Home_Or_Family_Matches,Inactive_Long_Term_Sick_Or_Disabled_Matches,Inactive_Other_Matches,Raw_Inactive_Home_Or_Family_Pct,Inactive_Home_Or_Family_Pct_Matches,Raw_Inactive_Long_Term_Sick_Or_Disabled_Pct,Inactive_Long_Term_Sick_Or_Disabled_Pct_Matches,Raw_Inactive_Other_Pct,Inactive_Other_Pct_Matches
0,E01009763,1637,164,73,70,10.018326,4.459377,4.276115,1637,164.0,...,True,True,True,True,10.018326,True,4.459377,True,4.276115,True
1,E01009764,1264,50,35,31,3.955696,2.768987,2.452532,1264,50.0,...,True,True,True,True,3.955696,True,2.768987,True,2.452532,True
2,E01009767,1265,69,66,28,5.454545,5.217391,2.213439,1265,69.0,...,True,True,True,True,5.454545,True,5.217391,True,2.213439,True
3,E01009769,1431,119,148,59,8.315863,10.342418,4.122991,1431,119.0,...,True,True,True,True,8.315863,True,10.342418,True,4.122991,True
4,E01009770,1214,73,69,50,6.013180,5.683690,4.118616,1214,73.0,...,True,True,True,True,6.013180,True,5.683690,True,4.118616,True


,LSOA_Code,Economic_Total_16_Plus,Inactive_Home_Or_Family,Inactive_Long_Term_Sick_Or_Disabled,Inactive_Other,Inactive_Home_Or_Family_Pct,Inactive_Long_Term_Sick_Or_Disabled_Pct,Inactive_Other_Pct,Raw_Economic_Total_16_Plus,Raw_Inactive_Home_Or_Family,...,Economic_Total_16_Plus_Matches,Inactive_Home_Or_Family_Matches,Inactive_Long_Term_Sick_Or_Disabled_Matches,Inactive_Other_Matches,Raw_Inactive_Home_Or_Family_Pct,Inactive_Home_Or_Family_Pct_Matches,Raw_Inactive_Long_Term_Sick_Or_Disabled_Pct,Inactive_Long_Term_Sick_Or_Disabled_Pct_Matches,Raw_Inactive_Other_Pct,Inactive_Other_Pct_Matches


#### Findings — Economic activity source verification

All 199 included LSOAs matched the original TS066 economic activity file. The population totals and counts for home or family responsibilities, long-term sickness or disability, and other inactivity matched the raw source exactly.

The three category percentages recalculated from the raw counts and economic activity population totals matched the saved percentages within the specified tolerance of 0.00000001 percentage points. No discrepancies were identified, so no corrections were required.

These checks verify the prepared source fields used to construct the selected-inactivity outcome in notebook 08. They do not constitute a separate check of notebook 08’s combined outcome calculation, and the selected categories must not be described as total economic inactivity.

#### 15.6 Verify selected deprivation values against the raw source

Compare the overall Index of Multiple Deprivation score, national rank and national decile with the original IoD2019 file for all 199 included LSOAs.

Ranks and deciles must match exactly. Scores use an absolute tolerance of 0.00000001 score units. These published measures are checked directly against the source, without recalculating them or using Census population denominators.

In [27]:
# Code developed with ChatGPT assistance.

# Reference: pandas read_csv documentation.
# https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html
# Used to load the source, whose header starts on the first line.
raw_iod = pd.read_csv(
    raw_dir / "dudley_iod.csv",
    encoding="utf-8-sig",
)

# Map the original headings to separate raw comparison fields.
iod_mapping = {
    "LSOA code (2011)": "LSOA_Code",
    "Index of Multiple Deprivation (IMD) Score": "Raw_IMD_Score",
    "Index of Multiple Deprivation (IMD) Rank "
    "(where 1 is most deprived)": "Raw_IMD_Rank",
    "Index of Multiple Deprivation (IMD) Decile "
    "(where 1 is most deprived 10% of LSOAs)": "Raw_IMD_Decile",
}

raw_iod = raw_iod[list(iod_mapping)].rename(columns=iod_mapping)

for column in ["Raw_IMD_Score", "Raw_IMD_Rank", "Raw_IMD_Decile"]:
    raw_iod[column] = pd.to_numeric(
        raw_iod[column], errors="raise"
    )

iod_columns = [
    "LSOA_Code",
    "IoD2019_IMD_Score",
    "IoD2019_IMD_Rank",
    "IoD2019_IMD_Decile",
]

# Match only prepared areas against the national source file.
# Reject duplicate keys rather than allowing repeated area records.
iod_check = prepared_check[iod_columns].merge(
    raw_iod,
    on="LSOA_Code",
    how="left",
    validate="one_to_one",
    indicator=True,
)

# Compare the published national rank and decile exactly.
iod_check["Rank_Matches"] = (
    iod_check["IoD2019_IMD_Rank"] == iod_check["Raw_IMD_Rank"]
)
iod_check["Decile_Matches"] = (
    iod_check["IoD2019_IMD_Decile"] == iod_check["Raw_IMD_Decile"]
)

# Reference: NumPy isclose documentation.
# https://numpy.org/doc/stable/reference/generated/numpy.isclose.html
# Used to allow only negligible floating-point score differences.
iod_check["Score_Matches"] = np.isclose(
    iod_check["IoD2019_IMD_Score"],
    iod_check["Raw_IMD_Score"],
    rtol=0,
    atol=1e-8,
)

iod_checks = [
    "Score_Matches",
    "Rank_Matches",
    "Decile_Matches",
]
iod_issues = iod_check.loc[
    ~iod_check[iod_checks].all(axis=1)
]

print("Areas checked:", len(iod_check))
print("Unmatched areas:", (iod_check["_merge"] != "both").sum())
print("Score mismatches:", (~iod_check["Score_Matches"]).sum())
print("Rank mismatches:", (~iod_check["Rank_Matches"]).sum())
print("Decile mismatches:", (~iod_check["Decile_Matches"]).sum())

display(iod_check.head())
display(iod_issues)

Areas checked: 199
Unmatched areas: 0
Score mismatches: 0
Rank mismatches: 0
Decile mismatches: 0


,LSOA_Code,IoD2019_IMD_Score,IoD2019_IMD_Rank,IoD2019_IMD_Decile,Raw_IMD_Score,Raw_IMD_Rank,Raw_IMD_Decile,_merge,Rank_Matches,Decile_Matches,Score_Matches
0,E01009763,31.273,7408,3,31.273,7408,3,both,True,True,True
1,E01009764,18.422,15760,5,18.422,15760,5,both,True,True,True
2,E01009767,24.422,11115,4,24.422,11115,4,both,True,True,True
3,E01009769,53.598,1656,1,53.598,1656,1,both,True,True,True
4,E01009770,27.311,9406,3,27.311,9406,3,both,True,True,True


,LSOA_Code,IoD2019_IMD_Score,IoD2019_IMD_Rank,IoD2019_IMD_Decile,Raw_IMD_Score,Raw_IMD_Rank,Raw_IMD_Decile,_merge,Rank_Matches,Decile_Matches,Score_Matches


#### Findings — Deprivation source verification

All 199 included LSOAs matched the original IoD2019 file. The overall IMD national ranks and deciles matched exactly, and the scores matched within the specified tolerance of 0.00000001 score units. No corrections were required.

These checks confirm that the selected published deprivation measures were preserved during preparation and merging. They do not verify every deprivation domain or reproduce the original deprivation calculations.

Selected source comparisons are now complete across all five datasets, with no unmatched areas or discrepancies in the fields checked.

---

## Conclusion

The five cleaned datasets have been combined using 199 shared LSOA codes. The official ONS lookup confirmed that these areas had unchanged boundaries between 2011 and 2021. All joins passed one-to-one validation, producing a prepared dataset with 199 rows and 105 columns.

Twenty-two percentage columns were calculated using the appropriate source-specific population or household totals. Original counts were retained, and deprivation fields were labelled with their 2019 release year.

Internal checks found no missing values, duplicate rows, duplicate LSOA codes, duplicate column names or infinite numeric values. All calculated percentages fall within 0–100.

Selected values were independently compared with the original raw files across all 199 included areas. These checks covered qualification totals and no-qualification counts, disability totals and limited-a-lot counts, household totals and no-car counts, economic activity totals and the three selected inactivity categories, and overall IMD scores, ranks and deciles.

All areas matched their source records. Checked counts, ranks and deciles matched exactly. Recalculated percentages and IMD scores matched within the specified absolute tolerance of 0.00000001 in their respective units. No discrepancies or required corrections were identified. These comparisons verify the selected fields, not every column or the accuracy of the original data collection.

The data dictionary documents all 105 prepared-data columns. A separate supplementary dictionary documents the two selected-inactivity variables created in notebook 08. Both dictionaries have been exported to `outputs/data_dictionary/`; the supplementary variables have not been added to the prepared CSV.

The combined dataset excludes four Census areas and two deprivation-only areas. Analysis must acknowledge this coverage limitation and the different source dates. The IoD2019 release contains measures whose underlying observation dates vary.

Economic activity percentages describe selected categories as shares of residents aged 16 and over. They do not provide an overall inactivity percentage or an unemployment rate. The selected-inactivity outcome created in notebook 08 excludes retirement and inactive students from its numerator, while retaining all residents aged 16 and over in its denominator.

The prepared dataset supports exploration of associations between health limitations, qualification deficits, car availability and selected economic inactivity across the included Dudley neighbourhoods. It supports a partial, association-based answer to the research question rather than proof that these barriers cause inactivity.

Area-level findings must not be used to infer individual circumstances or stigmatise neighbourhoods. Having no qualifications does not mean having no skills, disability does not establish inability or unwillingness to work, and having no car does not necessarily indicate poor transport access.

## Ethical Considerations for the Prepared Dataset

### Purpose and Responsible Use

The combined dataset should be used to explore how health limitations, qualification deficits and car availability are associated with selected economic inactivity across Dudley’s 199 included neighbourhoods. It should not be used to label residents, assign blame or determine individual eligibility for employment support.

Although the research question uses the word “drive”, the analysis must distinguish association from causation. Observed relationships cannot establish that a particular barrier causes economic inactivity.

### Geographic Coverage and Exclusions

Interpretation should acknowledge that the prepared dataset includes 199 areas with unchanged boundaries between 2011 and 2021. The four excluded Census areas and two deprivation-only areas must remain documented so that readers understand the limits of geographic coverage.

Findings should not be presented as complete coverage of all 203 Dudley Census 2021 neighbourhoods. Excluded areas should not be assumed to have the same characteristics as included areas.

### Different Source Dates

Census 2021 measures and IoD2019 deprivation measures should not be treated as observations collected at the same time. The underlying dates of deprivation indicators also vary.

Historical patterns should not be presented as a description of current conditions without further evidence.

### Population Definitions and Denominators

Each percentage should retain the denominator appropriate to its source. Economic activity, qualifications and disability measures refer to residents aged 16 and over, while car availability refers to households.

Differences between source population totals should remain visible rather than being replaced with a common total for convenience. Household percentages and person-based percentages should not be interpreted as describing the same population.

### Meaning of Selected Economic Inactivity

The selected-inactivity outcome should be clearly labelled wherever it is used. Its numerator combines home or family responsibilities, long-term sickness or disability, and other inactivity reasons. It excludes retirement and inactive students, while its denominator includes all residents aged 16 and over.

It should not be described as total economic inactivity, working-age inactivity or an unemployment rate. Caring responsibilities and other forms of inactivity should not automatically be framed as personal failings or circumstances that require intervention.

### Respectful Interpretation of Structural Barriers

Disability should not be interpreted as evidence of unwillingness or inability to work. Having no formal qualifications does not mean a person lacks skills or experience.

Households without a car should not automatically be described as experiencing transport poverty. Car availability alone does not capture public transport access, travel costs, service frequency or proximity to employment.

The conceptual overlap between health limitations and the sickness or disability component of the inactivity outcome should be acknowledged when interpreting their relationship.

### Area-Level Data and Privacy

Neighbourhood patterns should not be used to infer the circumstances of individual residents. The analysis should remain at the published aggregate level and should not attempt to identify people or combine the data with personal information.

Charts, dashboard labels and written findings should avoid stigmatising neighbourhoods or implying that all residents share an area’s average characteristics.

### Responsible Use of Deprivation Measures

Deprivation scores, rates, ranks and deciles should be interpreted according to their documented meanings. Composite scores should not be labelled as percentages, and national ranks should not be presented as rankings calculated only within Dudley.

Relative deprivation should not be treated as a complete description of a neighbourhood or the people who live there.

### Transparency and Validation

The original source files should be preserved, and transformations, exclusions, calculations and variable definitions should remain traceable through the notebooks and exported dictionaries.

Successful validation should not be presented as proof that the original data is error-free. The source comparisons cover selected fields across the included areas, rather than every column or every aspect of the original collection methodology.

The two supplementary inactivity variables should remain clearly distinguished from the 105 columns in the prepared CSV.

### Communication and Limitations

Findings should acknowledge influences not fully represented in the data, including age structure, caring responsibilities, employment opportunities and transport services. Neighbouring areas may also share characteristics, which should be considered when interpreting statistical results.

Source attribution, licensing information and the role of ChatGPT assistance should remain transparent. Conclusions should reflect the evidence and its limitations rather than imply certainty or recommend individual decisions that the data cannot support.